# 03 · Phân lớp (Classification) + Cross-Validation + F-score

## Mô tả bài toán
Phân lớp **nhị phân** từ đặc trưng giọng: dự đoán `status` (`0`=khỏe, `1`=Parkinson)
từ 22 feature (sau chuẩn hoá). Đánh giá bằng **10-fold Cross-Validation** và **F-score**,
chọn **đúng 3 thuật toán** từ danh sách: **KNN · Random Forest · Naive Bayes · AdaBoost · SVM**.

## Yêu cầu bài tập liên quan
- Train/test + k-fold CV (M05-T1), mô hình + F-score (M05-T2), báo cáo (M05-T3).

## Phạm vi của notebook này
- **M05** — thiết lập CV, chạy 3 mô hình, thu thập F-score, so sánh (không kết luận "tốt nhất").

## KHÔNG thuộc phạm vi
- Không tự điền F-score, không tự chọn mô hình tốt nhất (để người chạy + reviewer quyết).
- Không tái khám phá dữ liệu (→ notebook 01), không PCA mới (→ 02).


## Phân công (theo `TASK_ASSIGNMENTS.md`)

| Nhiệm vụ | Phụ trách | Reviewer |
|---|---|---|
| M05-T1 Train/test + k-fold CV (quy ước seed, k) | M05 | M01 |
| M05-T2 Mô hình phân lớp baseline + F-score | M05 | M01 (gate) + M06 |
| M05-T3 Báo cáo kết quả phân lớp | M05 | M06 |

**Tên / MSSV thành viên:** M05 = **Ngô Thị Anh Thư** (MSSV `087306008195`) (xem `TEAM.md`).


## Điều kiện bắt đầu

- **Đầu vào:** feature matrix 22 cột đã chuẩn hoá (M04-T1, từ notebook 02) + canonical CSV.
- **Phụ thuộc:** M04-T1. Nếu chưa có → đánh dấu **`Blocked`** (xem `HANDOFF.md` mục 6).
- ⚠️ **Hai điểm CẦN NHÓM XÁC MINH trước khi chạy (ghi trong `TASK_BOARD.md` / `docs/REPRODUCIBILITY.md`):**
  1. **Số fold k:** đề bài notebook này yêu cầu **k=10**, nhưng `docs/REPRODUCIBILITY.md` hiện ghi **k=5**.
     → Thống nhất (M05-T1) và ghi lại quyết định; notebook này dùng **k=10** theo đề bài.
  2. **Loại F-score:** tài liệu chưa chốt **macro** hay **weighted** (nhìn chung nên chọn macro vì bài
     phân lớp cân bằng). → Xác minh trong nhóm (M05-T2 / M06).


In [ ]:
# TODO (M05-T1): nạp feature matrix + nhãn (từ notebook 02)
# import pandas as pd
# import numpy as np
# df = pd.read_csv("data/processed/parkinsons_v1.csv")
# y = df["status"]
# X = df[[c for c in df.columns if c != "status"]]   # 22 feature
pass

## Bước 1 · Xác định biến mục tiêu
**Mục tiêu:** chốt `y = status` (nhị phân 0/1).
**Câu hỏi tự kiểm:** `y` có đúng 2 lớp (0 và 1)? Có NaN trong `y` không?


In [ ]:
# TODO (M05-T1): xác nhận y là nhị phân và sạch
# y.unique()        # mong đợi [0, 1]
# y.isna().sum()   # mong đợi 0
pass

## Bước 2 · Chuẩn bị đặc trưng và nhãn
**Mục tiêu:** `X` (22 feature, số) và `y` (0/1), cùng thứ tự hàng.
**Lưu ý (QUY TRÌNH):** mọi bước **học tham số** (scaler, impute, PCA...) phải được
**fit trên mỗi fold tập huấn** và **transform trên fold kiểm tra** — KHÔNG fit trên toàn bộ dữ liệu.


In [ ]:
# TODO (M05-T1): xây X, y; chuẩn bị để nạp vào Pipeline (SINH VIÊN TỰ TRIỂN KHAI)
# X, y  # (đã chuẩn hoá nếu theo M04-T1)
# # Gợi ý: dùng sklearn.pipeline.Pipeline([("scaler", ...), ("model", ...)])
# #         để tiền xử lý & mô hình nằm CÙNG một bước -> tránh data leakage
pass

## Bước 3 · Thiết kế quy trình tiền xử lý (tránh data leakage)
**Mục tiêu:** quy ước "làm đúng" để kết quả không bị phồng lên do leakage.
**Quy tắc (bắt buộc đọc):**
- Scaler / imputer / PCA: **fit on train fold**, **transform on validation fold**.
- Tinh chỉnh tham số (grid/random search): chỉ trên **nội bộ fold train** (inner CV).
- **Test set chỉ dùng MỘT LẦN** ở cuối, sau khi đã chọn xong tham số.


## Bước 4 · Thiết lập 10-fold Cross-Validation
**Mục tiêu:** đánh giá ổn định trên dataset nhỏ (195 mẫu) nhờ CV thay vì 1 lần train/test.
**Quy ước:** `StratifiedKFold(n_splits=10)` (giữ tỉ lệ 0/1 mỗi fold), cố định `random_state` (seed).
> ⚠️ Theo đề bài notebook này dùng **k=10** (xem lưu ý "CẦN XÁC MINH" ở §Điều kiện bắt đầu).


In [ ]:
# TODO (M05-T1): tạo CV (SINH VIÊN TỰ TRIỂN KHAI)
# from sklearn.model_selection import StratifiedKFold, cross_validate
# cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)   # seed theo REPRODUCIBILITY.md
# # GHI LẠI: số fold, seed -> results/ + REPRODUCIBILITY.md
pass

## Bước 5 · Chọn ĐÚNG 3 thuật toán (từ danh sách)
**Mục tiêu:** chọn **đúng 3** trong {KNN, Random Forest, Naive Bayes, AdaBoost, SVM} để so sánh.
**Làm gì:** chọn 3 + ghi lý do (vd KNN làm baseline, RF + SVM là hai lựa chọn mạnh). "Đúng 3" nghĩa là
đủ 3, không phải cả 5.


In [ ]:
# TODO (M05-T2): khai báo 3 mô hình (SINH VIÊN CHỌN 3/5)
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
# from sklearn.naive_bayes import GaussianNB
# from sklearn.svm import SVC
# models = { ... }   # sinh viên điền ĐÚNG 3 trong 5 cái ở trên, kèm vài tham số ban đầu
pass

## Bước 6 · Xác định phạm vi tinh chỉnh tham số
**Mục tiêu:** đặt ra **vài giá trị thử** cho 1–2 tham số chính của từng mô hình (không chạy lưới lớn).
**Ví dụ gợi ý (không bắt buộc):** KNN: `n_neighbors` ∈ {3,5,7}; RF: `n_estimators` ∈ {50,100};
SVM: `C` ∈ {0.1,1,10}; NB/AdaBoost: dùng mặc định hoặc 1–2 biến.
**Lưu ý:** tinh chỉnh chỉ trên **fold train** (inner CV), không chạm test.


In [ ]:
# TODO (M05-T2): xác định bộ tham số sẽ thử (SINH VIÊN TỰ TRIỂN KHAI)
# # ví dụ: dùng GridSearchCV(n_splits=5, scoring="f1_macro", cv=inner_cv) bên TRONG fold train
# # KHÔNG được dùng test set để chọn tham số
# params = { "KNN": {"n_neighbors": [3,5,7]}, "RF": {"n_estimators": [50,100]}, "SVM": {"C": [0.1,1,10]} }
pass

## Bước 7 · Thu thập F-score
**Mục tiêu:** chạy CV, thu **F-score** (macro hay weighted — **xác minh trước**, xem §Điều kiện bắt đầu)
theo từng mô hình, cả **mean ± std** trên các fold.
**Lưu ý:** **KHÔNG** tự điền số vào bảng dưới — chạy thật rồi ghi.


In [ ]:
# TODO (M05-T2): chạy CV + lấy F-score (SINH VIÊN TỰ TRIỂN KHAI)
# results = cross_validate(model, X, y, cv=cv, scoring="f1_macro", return_train_score=False)
# results["test_score"]   # sinh viên tính mean ± std rồi GHI vào bảng (đừng viết sẵn)
# # Ghi rõ: F-score macro hay weighted? -> trong results/ + REPRODUCIBILITY.md
pass

## Kết quả & nhận xét (sinh viên tự điền — KHÔNG điền sẵn / KHÔNG kết luận "tốt nhất")

**Bảng so sánh (điền KHI CHẠY THẬT):**

| Mô hình | F-score (CV) mean ± std | Ghi chú tham số |
|---|---|---|
| _(mô hình 1)_ | _điền_ | _điền_ |
| _(mô hình 2)_ | _điền_ | _điền_ |
| _(mô hình 3)_ | _điền_ | _điền_ |

**Câu hỏi gợi ý:**
1. Mô hình nào có F-score cao nhất **trên CV**? Chênh lệch có đáng kể so với std không?
2. Có mô hình nào "dễ" overfit (F-score train cao, test thấp) không?
3. Nếu dùng F-score weighted thay macro, bảng kết quả có đổi thứ hạng không? Vì sao?
> **Không viết "mô hình X tốt nhất"** ở đây để reviewer cùng kết luận (M05-T3 / M06).


## Checklist nghiệm thu (trạng thái ban đầu: `Todo`)

### Hoàn thành
- [ ] Đã chốt `y=status`, `X` = 22 feature (không `name`/`status`).
- [ ] Đã thiết lập **10-fold stratified CV** (ghi seed + k).
- [ ] Đã chọn **đúng 3** thuật toán trong 5 cho phép.
- [ ] Đã chạy F-score (macro/weighted) **mean ± std** trên CV.
- [ ] Đã ghi so sánh 3 mô hình vào bảng + `results/`.

### Kiểm tra anti-leakage (BẮT BUỘC)
- [ ] Scaler/PCA/param **fit trên train fold**, transform trên val fold (dùng Pipeline).
- [ ] Tinh chỉnh tham số **không** dùng test set.
- [ ] Test set (nếu có) chỉ dùng **một lần** ở cuối.

### Bàn giao
- [ ] Đã viết `results/classification_report.md` (M05-T3).
- [ ] Đã chuyển M05-T1..T3 sang `Review`; thông báo M01 (gate) + M06.
- [ ] Đã đánh dấu (nếu chưa) 2 điểm CẦN XÁC MINH: **k (10 vs 5)** và **F-score macro/weighted**.

### Vấn đề chưa giải quyết
- [ ] _(ghi nếu có)_


## Tái lập & xử lý Blocked

- **Nguồn dữ liệu:** canonical CSV `data/processed/parkinsons_v1.csv` + feature matrix chuẩn hoá (M04-T1).
- **Đầu ra dự kiến:** `results/classification_report.md` (F-score 3 mô hình theo CV).
- **Ghi lại** (vào `results/` + `docs/REPRODUCIBILITY.md`): seed, k=10 (theo đề bài), loại F-score,
  3 mô hình + tham số, phiên bản thư viện.
- **Không hardcode** đường dẫn cá nhân (chạy từ gốc project / Colab).
- **Blocked:** thiếu feature matrix (M04-T1) hoặc 2 điểm chưa xác minh (k, F-score) → ghi `Blocked`
  trong `TASK_BOARD.md`, nêu rõ; thông báo M01 (điều phối). Xem `HANDOFF.md` mục 6.
